### This notebook give you a minimal code to generate synthetic time series using the pretrained models after running stage1 and 2.

Prerequisite
- `stage1-Wind_model.ckpt` and `stage2-Wind_model.ckpt` must exist in `saved_models/`.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import torch
from utils import load_yaml_param_settings
import numpy as np
import torch
from preprocessing.preprocess import dataset_importer_ws

In [ ]:
# settings
dataset_name = 'Wind'
gpu_device_idx = 0

# fetch necessary data
config = load_yaml_param_settings(os.path.join('configs', 'config.yaml'))
config2 = load_yaml_param_settings(os.path.join('configs', 'config_consecutive_wsm.yaml'))
input_length = int(1440)
n_classes = int(1)

In [ ]:
device1 = torch.device('cpu')
device2 = torch.device('cuda')

# Independent sampling

## Weather as Features

In [ ]:
# load a model wrapper for the evaluation
# !!! change the mean and std to None if not scaling
# !!! be careful for the dataset_name here you might use different name for stage 2
from evaluation.evaluation import Evaluation
evaluation = Evaluation('Wind_wf',
                        'Wind_wf',
                        3,
                        input_length,
                        n_classes,
                        False,
                        False,
                        False,
                        gpu_device_idx,
                        config,
                        ).to(gpu_device_idx)

In [ ]:
num_samples = 50

with torch.no_grad():
  evaluation.stage2.eval()
  xhat = evaluation.sample(num_samples, 'unconditional', unscale=False, weather_states=None)

xhat = xhat.detach().cpu().numpy()
xhat.shape

In [ ]:
# save synthetic data
# os.makedirs('output_synthetic/', exist_ok=True)
# np.savez('output_synthetic/wf_independent', data=xhat)

## Weather as Embeddings

In [ ]:
from evaluation.evaluation import Evaluation
evaluation = Evaluation('Wind_wsm',
                        'Wind_wsm',
                        2,
                        input_length,
                        n_classes,
                        True,
                        False,
                        True,
                        gpu_device_idx,
                        config,
                        ).to(gpu_device_idx)

In [ ]:
# use weather states from training_imputed_full_days_ws
X_train = dataset_importer_ws.X_train[:, :, :1440]
X_test = dataset_importer_ws.X_test[:, :, :1440]
X_train_full = np.concatenate((X_train, X_test), axis=0)

In [ ]:
device1 = torch.device('cpu')
device2 = torch.device('cuda')

ws_train = dataset_importer_ws.ws_train
ws_test = dataset_importer_ws.ws_test
ws_train_full = np.concatenate((ws_train, ws_test), axis=0)
ws_ = torch.from_numpy(ws_train_full).long().to(device2)
ws_.shape

In [ ]:
plt.figure(figsize=(12,5))
plt.plot(X_train_full[0, :, :].T)
plt.plot(ws_.cpu()[0, :].T, alpha=0.5)

In [ ]:
num_samples = 50
with torch.no_grad():
  evaluation.stage2.eval()
  xhat, wshat = evaluation.sample(num_samples, 'unconditional', unscale=False, weather_states_mask=True)

In [ ]:
plt.figure(figsize=(12,5))
plt.plot(xhat[0, :, :].T)
plt.plot(wshat.cpu()[0, :].T, alpha=0.5)

In [ ]:
xhat = xhat.detach().cpu().numpy()

# Consecutive Sampling

## Weather as Features

In [ ]:
# !!! be careful for the dataset_name here you might use different name for stage 2
from evaluation.evaluation import Evaluation
evaluation = Evaluation('Wind_wf',
                        'Wind_consecutive_wf',
                        3,
                        input_length,
                        n_classes,
                        False,
                        True,
                        False,
                        gpu_device_idx,
                        config,
                        ).to(gpu_device_idx)

In [ ]:
X_org = np.load("datasets/training_imputed_full_days_ws.npz")['data']
X_org.shape

x = torch.from_numpy(X_org).float().to(device2)

evaluation.stage1.eval()
x_rec, z_q, s_ = evaluation.stage1.forward(batch=(x, None), batch_idx=-1, return_x_vq=True)

# num_days = x_rec.shape[0]
# print('Use number of real days:', num_days)

In [ ]:
def consecutive_simulation(start_day):
  s_new = s_.clone()[start_day].reshape(1, -1)
  device =  torch.device('cuda')

  s_new_lst = []
  xhat_lst = []

  for d in range(21):

    s_before = s_new[0].to(device)
    xhat, _, s_new = evaluation.sample(1, 'consecutive', unscale=False, return_representations=True, s_before=s_before)
    s_new_lst.append(s_new[0])
    xhat_lst.append(xhat[0])

  concatenated = torch.cat(xhat_lst, dim=1)
  return concatenated

In [ ]:
consecutive_lst = []

for day in range(3):
  consecutive_days = consecutive_simulation(day)
  consecutive_lst.append(consecutive_days)

In [ ]:
consecutive_samples_wf = torch.stack(consecutive_lst, dim=0)
print(consecutive_samples_wf.shape)
# np.savez("output_synthetic/consecutive_samples_wf.npz", data=consecutive_samples_wf.detach().cpu().numpy())

## Weather as Embeddings

In [ ]:
from evaluation.evaluation import Evaluation
evaluation = Evaluation('Wind_wsm',
                        'Wind_consecutive_wsm',
                        2,
                        input_length,
                        n_classes,
                        True,
                        True,
                        True,
                        gpu_device_idx,
                        config2,
                        ).to(gpu_device_idx)

In [ ]:
X_org = np.load("datasets/training_imputed_full_days_ws.npz")['data']
X_org.shape

x_wv = X_org[:, :2, :]
x_ws = X_org[:, 2, :]

x = torch.from_numpy(x_wv).float().to(device2)
ws_ = torch.from_numpy(x_ws).long().to(device2)

evaluation.stage1.eval()
x_rec, z_q, s_ = evaluation.stage1.forward(batch=(x, None), batch_idx=-1, return_x_vq=True)

In [ ]:
def consecutive_simulation_wsm(start_day):
  
  device =  torch.device('cuda')
  s_new = s_.clone()[start_day].reshape(1, -1)
  ws_new = ws_.clone()[start_day].reshape(1, -1)

  s_new_lst = []
  xhat_lst = []
  ws_new_lst = []

  for d in range(21):

    s_before = s_new[0].to(device)
    ws_before = ws_new[0].to(device)
    xhat, _, s_new, ws_new = evaluation.sample(1, 'consecutive', unscale=False, return_representations=True, s_before=s_before, ws_before=ws_before, weather_states_mask=True)
    s_new_lst.append(s_new[0])
    xhat_lst.append(xhat[0])
    ws_new_lst.append(ws_new[0])

  concatenated = torch.cat(xhat_lst, dim=1)
  return concatenated

In [ ]:
consecutive_lst = []

for day in range(2):
  consecutive_days = consecutive_simulation_wsm(day)
  consecutive_lst.append(consecutive_days)

In [ ]:
consecutive_samples_sfr_wsm = torch.stack(consecutive_lst, dim=0)
print(consecutive_samples_sfr_wsm.shape)
# np.savez("output_synthetic/consecutive_samples_sfr_wsm.npz", data=consecutive_samples_sfr_wsm.detach().cpu().numpy())

start from unconditional days, do independent sampling first

In [ ]:
# read the unconditional synthetic days
xhat_unconditional  = np.load('output_synthetic/unconditional_synthetic_wsm.npz')['data']
xhat_unconditional.shape

In [ ]:
# sample n as the first day in the consecutive days

# Randomly sample n indices
sample_indices = np.random.choice(xhat_unconditional.shape[0], size=2*10, replace=False)

# Sample the data
xhat_unconditional_samples = xhat_unconditional[sample_indices]
xhat_unconditional_samples.shape

In [ ]:
x = torch.from_numpy(xhat_unconditional_samples[:, :2, :]).float().to(device2)
evaluation.stage1.eval()
x_rec, z_q, s_ = evaluation.stage1.forward(batch=(x, None), batch_idx=-1, return_x_vq=True)

In [ ]:
ws_ = torch.from_numpy(xhat_unconditional_samples[:, 2, :]).long().to(device2)
ws_.shape

In [ ]:
def consecutive_simulation_wsm(start_day):
  
  device =  torch.device('cuda')
  s_new = s_.clone()[start_day].reshape(1, -1)
  ws_new = ws_.clone()[start_day].reshape(1, -1)

  s_new_lst = []
  xhat_lst = []
  ws_new_lst = []

  for d in range(21):

    s_before = s_new[0].to(device)
    ws_before = ws_new[0].to(device)
    xhat, _, s_new, ws_new = evaluation.sample(1, 'consecutive', unscale=False, return_representations=True, s_before=s_before, ws_before=ws_before, weather_states_mask=True)
    s_new_lst.append(s_new[0])
    xhat_lst.append(xhat[0])
    ws_new_lst.append(ws_new[0])

  concatenated = torch.cat(xhat_lst, dim=1)
  return concatenated

In [ ]:
consecutive_lst = []

for day in range(2):
  consecutive_days = consecutive_simulation_wsm(day)
  consecutive_lst.append(consecutive_days)

In [ ]:
consecutive_samples_sfg_wsm = torch.stack(consecutive_lst, dim=0)
print(consecutive_samples_sfg_wsm.shape)
# np.savez("output_synthetic/consecutive_samples_sfg_wsm.npz", data=consecutive_samples_sfg_wsm.detach().cpu().numpy())
# np.savetxt('output_synthetic/consecutive_samples_sfg_wsm.csv', consecutive_samples_sfg_wsm.permute(0, 2, 1).reshape(-1, 2).detach().cpu().numpy(), delimiter=',')